In [ ]:
/**
 * Scaling to Thousands of Users & Beyond:
 * > Now that we've handled the basics, next phase is all about handling traffic, 
 *   concurrency, and distribution.
 * > This is where systems need to stop being "toy projects" and start acting like
 *   production-grade architectures.
 *   - Application-Level Scaling (Horizontal vs Vertical)
 *   - Data-Layer Scaling (Sharding and Replication)
 *   - Microservices and Monolith Architecture - trade-offs and pitfalls
 *   - Single point of failure and system bottlenecks
 *   - Consistency in distributed systems (CAP theorem in real life)
 *   - Caching 101
*/

In [ ]:
/**
 * Database Layer Scaling:
*/

In [ ]:
/**
 * Scale Database Architecture:
 * > Jaise-jaise users aur traffic badhta hai, database bottleneck ban jaata hai.
 * > Agar ek hi database use karte hain:
 *   - Har request (READ + WRITE) wahi jaayegi
 *   - High CPU / IOPS usage
 *   - Slow queries
 *   - System latency increase
 *   - Eventually system crash / downtime
 * 
 * Solution: Read-Write Splitting
 * 1. Write Operations
 *    - Insert/Update/Delete 
 *    - Always Write DB m jaati hain
 * 2. Read Operations
 *    - Select Queries
 *    - Read DBs handle karte hai
 * 3. Data Replication
 *    - Write DB apna data Read DBs ko copy (replicate) karta hai
 *    - Ye sync asynchronous hota hai
 * 
 *         +---------------------------+
 *         | Load Balancer - NGINX/ELB |
 *         +---------------------------+
 *                       |
 *             +---------+---------+
 *             |                   |
 *             V                   V
 *         +--------+         +--------+
 *      +--| Server |         | Server |+--+
 *      |  +--------+         +--------+   |
 *      |      |                   |       |
 *      |      +---------+---------+       |
 *      |                | WRITE ONLY (Source of Truth)
 *      |                V                 |
 *      |           +----------+           |
 *      |  +--------| Database |--------+  |
 *      |  |        +----------+        |  |   # Eventual Consistency means:
 *      |  |                            |  |            |
 *      V  V                            V  V            V
 *   +----------+                 +----------+  - WRITE DB aur READ DB ke beech immediate 
 *   | Database |                 | Database |    consistency nahi hoti 
 *   +----------+                 +----------+  - Delay hota hai: WRITE DB → (some delay) → READ DB
 *    (READ Only)                  (READ Only)  - Example:
 *                                                - User ne payment ki
 *                                                - WRITE DB updated instantly
 *                                                - READ DB me update thoda late reflect hoga
 *                                                - This is called Eventual Consistency. 
 * 
 * > Agar WRITE DB crash ho jaata hai, to usse system se hata diya jaata hai
 *   aur kisi ek READ DB ko promote karke main (WRITE) database bana diya jaata hai.
 * > Ye switch automatically kuch seconds ke andar ho jaata hai (failover).
 * > Agar koi machine baar-baar fail hoti hai, to usse system se remove kar diya jaata 
 *   hai, aur ek nayi machine create ki jaati hai jo kisi existing READ DB (replica) se 
 *   data copy karti hai.
 * 
 * Note: Always use 'Point-in-time' recovery in database.
 */

In [ ]:
/**
 * Bonus Concepts and Case Studies:
*/

In [ ]:
/**
 * Case Study: Phonepay
 * > Humein pata hai ki PhonePe me do databases use ho rahe hain, lekin WRITE sirf Mumbai
 *   wale DB me hota hai. Dono databases aapas me connected hote hain aur continuously ek
 *   dusre ko ping bhejte rehte hain taaki health status check hota rahe.
 * > Hyderabad Database READ DB ka role play karta hai, lekin kya ye WRITE DB banne ke 
 *   liye ready hai? Haan! 
 * > Mumbai DB me jo bhi data aata hai, wo Hyderabad DB me bhi continuously copy 
 *   (replicate) hota rehta hai. Isliye Hyderabad DB almost up-to-date rehta hai aur 
 *   zarurat padne par turant WRITE handle kar sakta hai.
 * > Mumbai DB ko humne LEADER (Primary) bana rakha hai kyunki saare write operations wahi
 *   handle ho rahe hain. Hyderabad DB baar-baar Mumbai DB ko ping karta rehta hai taaki 
 *   confirm kar sake ki wo alive hai ya nahi.
 * > Agar Mumbai DB ping ka response dena band kar deta hai, to system ye assume karta hai
 *   ki Mumbai DB down ho gaya hai. Aise case me leadership automatically Mumbai se 
 *   Hyderabad shift kar di jaati hai, aur PhonePe ke saare write operations Hyderabad DB
 *   par hone lagte hain.
 *                 
 *                                 +----------+   
 *                                 | PhonePay |
 *                                 +----------+
 *                         Mumbai        |      Hyderabad
 *                       +---------------+---------------+
 *                       |                               |
 *                       V                               V
 *                  +----------+                    +----------+
 *                  | Database |<------------------>| Database |
 *                  +----------+                    +----------+
 * 
 * Real World Situation:
 * > PhonePe me Mumbai ka database primary hai jahan sab write ho raha hai, aur Hyderabad
 *   database usko continuously ping karke check karta rehta hai ki wo alive hai ya nahi.
 *   Lekin maan lo ye ping wali connection line hi crash ho jaati hai kisi network issue
 *   ki wajah se.
 * > Ab problem ye hoti hai ki Hyderabad DB ko Mumbai se response nahi milta, to wo assume
 *   karta hai ki Mumbai DB down ho gaya hai aur khud ko leader declare kar deta hai.
 *   Dusri taraf Mumbai DB actually alive hai, to wo bhi khud ko leader hi maanta rehta hai.
 *   Yani dono databases ek saath leader ban jaate hain, jo ki normal crash se bhi zyada
 *   dangerous situation hai.
 * > Ab ek real example lete hain: maan lo Piyush ne ₹400 receive kiye aur wo Mumbai DB me
 *   update ho gaya. Uske baad Piyush ne ₹1200 spend kiye, lekin kisi wajah se request
 *   Hyderabad DB pe chali gayi. Hyderabad DB ke paas to updated data tha hi nahi (₹400 ka),
 *   to uske hisaab se Piyush ke paas paise hi nahi the. Agar system Hyderabad DB se operate
 *   kare, to transaction ka behavior galat ho sakta hai — ya to paise cut hi nahi honge,
 *   ya inconsistent state create ho jaayegi.
 * > Ab aur problem tab badh jaati hai jab baad me Mumbai aur Hyderabad ke beech connection
 *   wapas se theek ho jaata hai. Dono databases apne-apne hisaab se sahi lag rahe hote hain,
 *   aur dono ne alag-alag data write kar diya hota hai. Ab jab sync karne ki koshish karenge,
 *   to sawal ye uthta hai: kaunsa data correct hai? Dono hi databases “healthy” dikh rahe
 *   hain, lekin data inconsistent ho chuka hai.
 * > Isi situation ko "Split Brain Problem" kehte hain. Ye distributed systems ka ek 
 *   classic issue hai, aur financial systems (jaise PhonePe) is problem se sabse zyada 
 *   darte hain kyunki yahan data inconsistency ka matlab direct paiso ka loss ya 
 *   corruption ho sakta hai.
 * 
 * Solution-1: Single Region Critical Write
 * > Idea: Critical writes (important operations jaise payment, balance update)
 *   sirf ek hi fixed region me honge.
 * > Example: 
 *   - Mumbai = WRITE region
 *   - Hyderabad = Sirf READ/Backup 
 * > Chahe user kahi se bhi aaye, critical write humesha Mumbai me hi hoga
 * 
 * > Kaam Kaise krta hai?
 *   - User India ke kisi bhi city se request bheje
 *   - System us request ko route karega Mumbai region me
 *   - Wahi pe:
 *        - Lock lagega
 *        - Processing hogi
 *        - Final write hoga
 *   - Baaki regions:
 *        - Sirf read karenge
 *        - Ya async sync se update honge
 * 
 * > Pehle kya issue tha:
 *   - Mumbai aur Hyderabad dono write kar rahe the
 *   - Network issue me dono alag-alag state me chale gaye
 * > Ab kya change hua:
 *   - Sirf ek region (Mumbai) ko write ka right mila
 *   - Dusra region kabhi write nahi karega
 * > Isliye:
 *   - Conflict ka chance khatam
 *   - Duplicate write ka risk kam
 * 
 * > Limitation:
 *   - Agar Mumbai region down ho gaya: System temporarily write nahi kar paayega
 *   - Isliye: 
 *     - High Consistency milti hai
 *     - Lekin availability thodi sacrifice hoti hai
 * 
 * Note: "Critical operations sirf ek trusted region me karna, taaki multiple jagah se
 *        write conflict na ho". And this approach is called Prefer Unavailability over
 *        Inconsistency, and this is the most preferred way.
 * 
 * 
 * Solution-2: Quorum - Majority Voting
 * > Split Brain problem me issue ye tha ki:
 *   - Mumbai aur Hyderabad dono khud ko leader maan rahe the
 *   - Dono independently WRITE kar rahe the
 *   - System decide hi nahi kar pa raha tha kaun sahi hai
 * > Yani problem thi: "single source of truth missing"
 * 
 * > Quorum bolta hai:
 *   - Minimum 3 nodes/databases hone chahiye. Rule simple hai:
 *     Jab tak majority (matlab 3 me se kam se kam 2 nodes) approval nahi denge,
 *     tab tak koi WRITE operation perform nahi hoga.
 *   - Matlab:
 *     - Ek single DB khud se decision nhi le sakta
 *     - 3 me se 2 nodes bolnge "haan" tabhi WRITE hoga
 * > Example:
 *   Agar ek request aati hai aur system WRITE karna chahta hai, to usse pehle
 *   3 me se 2 nodes ka approval lena padega. Agar majority agree nahi karti,
 *   to WRITE reject kar diya jaayega. Isse consistency maintain karne me help milti hai.
 * 
 * Example:
 * > Pehle kya ho raha tha:
 *   - Mumbai bol raha tha "main leader hu"
 *   - Hyderabad bol raha tha "main leader hu"
 *   - Dono WRITE kar rahe the → data conflict
 * > Ab Quorum ke saath kya hoga:
 *   - Total 3 nodes hain: Mumbai, Hyderabad, Bangalore
 * 
 *   a. Case-1: Normal Situation
 *      > Mumbai WRITE karna chahta hai
 *      > Wo Hyderabad + Bangalore se approval maangega
 *      > Agar 2 nodes agree kar gaye → WRITE allowed
 * 
 *   b. Network issue (Split Brain jaisa)
 *      > Mumbai aur Hyderabad ka connection toot gaya
 *      > Ab Mumbai sirf Bangalore se baat kr paa rha hai:
 *        - 2 nodes mil gaye
 *        - Majority mil gya
 *        - Mumbai WRITE kar sakta hai
 *      > Hyderabad akela reh gya:
 *        - Uske paas sirf 1 vote hai
 *        - Majority nahi mili
 *        - Hyderabad WRITE nhi kar sakta
 * 
 * > Final Impact:
 *   - Sirf ek side WRITE karega (jiske paas majority hai)
 *   - Dusra side automatically inactive ho jaayega (no WRITE)
 *   - Isse:
 *     a. Do leaders ka problem khatam
 *     b. Data conflict nahi hota
 *     c. System me ek hi source of truth rehta hai
 * 
 * Note: "Quorum ensure karta hai ki system me kabhi bhi ek se zyada
 *        leader active na ho, kyunki WRITE ke liye majority zaroori hai."
*/

In [ ]:
/**
 * Problem: Lock + Timing Issue
 * > Ab tak humne Quorum use karke ek important problem solve kar li:
 *   - Split brain khatam ho gaya
 *   - Ek hi leader WRITE karega (majority rule se)
 * > Yani:
 *   - Ab system me ek hi source of truth hai
 *   - Multiple leaders wala issue resolve ho gaya
 * > But ab new problem aata hai:
 *   Quorum ne "kaun WRITE karega" solve kiya, lekin "WRITE kaise safe hoga" ye abhi bhi
 *   problem hai. Yahi par locking + timing issue aata hai.
 *
 * > Maan lo ab system stable hai: Leader decide ho chuka hai (Quorum se)
 * > Ab user "Buy Now" click karta hai:
 *   - Leader DB lock leta hai (taaki duplicate write na ho)
 *   - Processing start hoti hai
 * > Yaha tak sab controlled hai, lekin beech m garbage collector (GC) aata hai aur
 *   process ko HALT kar deta hai:
 *   - Lock logically exist karta hai
 *   - Lekin actual process ruk gaya
 *   - GC ka kaam memory clean karna hota hai, aur kabhi-kabhi system ko stable rakhne ke 
 *     liye sab running processes ko thodi der ke liye pause (HALT) karna padta hai
 *     (isse "stop-the-world" pause kehte hain).
 *     GC ye nahi dekhta ki process ne lock liya hai ya nahi.
 * > User ko lagta hai transaction hua nahi, wo dobara "Buy Now" click karta hai.
 * 
 * > Ab system ka behavior:
 *   - Second request aati hai
 *   - Leader DB ise new request samajh leta hai
 *   - Transaction execute ho jata hai
 * > Thodi der baad:
 *   - Pehla halted process resume hota hai
 *   - Wo bhi transaction complete kar deta hai
 *
 * > Final Result:
 *   Result ye hota hai ki jo ₹1200 ka payment ek baar hona chahiye tha, wo do baar ho 
 *   jaata hai aur total ₹2400 cut jaata hai, jabki:
 *   - Leader ek hi tha (Quorum sahi tha)
 *   - Phir bhi duplicate transaction ho gya, because:
 *     a. System duplicate intent detect nhi kar paaya
 *     b. Lock timing issue ki wajah se unreliable ho gya
 * > Quorum solve krta hai: "Kaun WRITE karega"
 *   Lekin solve nhi karta: "Ek WRITE sirf ek baar hi ho"
 * 
 * 
 * Solution-1: Fencing Token - "Old process ko rokna"
 * > Fencing token ka main kaam hai stale(puraane) process ko block karna.
 * > Jab distributed system me lock use hota hai aur koi process HALT ho jaata hai
 *   (jaise GD ki wajah se), to wo baad me resume hoke galat write kar sakta hai.
 * > Yha afencing token help karta hai:
 *   - Har lock ke saath ek increasing token milta hai
 *   - Naya process   -> bada token
 *   - Purana process -> chhota token
 * > Database Rule:
 *   - Sirf latest (highest token) accept hoga
 *   - Old token -> reject
 * > Focus: Concurrency Control (Race Condition / Stale Writes)
 * 
 * Solution-2: Idempotency - "Same request ko repeat hone se rokna"
 * > Idempotency ka main kaam hai same user action ko multiple baar execute hone se rokna.
 * > Jab user:
 *   - double click karta hai
 *   - retry hota hai (network issue)
 *   to system same request ko dobara process kar sakta hai. 
 * > Yaha idempotency use hoti hai:
 *   - Har request ko ek unique ID milti hai (txn_id)
 *   - Agar same ID dobara aaye -> ignore
 * > Focus: Duplicate Requests (same intent)
 */

In [ ]:
/**
 * Problem: Leader Election in Distributed Database
 * 
 * > Now suppose humare paas multiple DBs hai:
 *                                  +----------+   
 *                                  | PhonePay |
 *                                  +----------+
 *                                        |
 *           +--------------+-------------+---------------+---------------+
 *           |              |             |               |               |
 *           V              V             V               V               V
 *    +------------+  +------------+  +------------+  +------------+  +------------+
 *    | Database-1 |  | Database-2 |  | Database-3 |  | Database-4 |  | Database-5 |
 *    +------------+  +------------+  +------------+  +------------+  +------------+
 *       Leader
 * 
 * > Ab inn DBs me se leader kaun hai? Maan lo DB-1 ko leader maan liya.
 * > Ab problem:
 *   - Baaki DBs ko kaise pata chalega ki leader kaun hai?
 *   - Har DB ko DB-1 ko ping karke check karna padega
 * > Isse:
 *   - System complex ho jata hai
 *   - Har node ko manually track karna padta hai
 * > Aur bada problem:
 *   - Agar DB-1 down ho gaya to?
 *   - Naya leader kaise choose hoga?
 *   - Kaun decide karega ki ab DB-2 leader bane ya DB-3?
 * > Yani: Leader selection ka koi reliable mechanism nahi hai
 *
 * 
 * Solution: Central Coordination
 * > Leader election ko manage karne ke liye ek central coordination service use hoti hai
 *   jo voting process ko organize karti hai.
 * > Ye service sabhi databases (nodes) ko clearly batati hai:
 *   - Current leader kaun hai
 *   - Agar leader down ho jaye to naya leader kaun banega (voting ke through)
 * > Flow simple hai:
 *   - Sab DBs central service me register karte hain
 *   - Voting hoti hai (majority based)
 *   - Ek leader choose hota hai
 *   - Baaki nodes us leader ko follow karte hain
 * > Agar leader crash ho jaye:
 *   - Service automatically detect karti hai
 *   - Dobara voting hoti hai
 *   - Naya leader select ho jata hai
 * > Important:
 *   - DBs ko ek dusre ko ping karne ki zarurat nahi padti
 *   - Coordination ka kaam central service handle karti hai
 *
 * > Zookeeper aur etcd are both popular distributed coordination systems used for
 *   leader election, configuration management, and service discovery.
*/

In [ ]:
/**
 * Problem:
 *      1        2        3        4
 *   +-----+  +-----+  +-----+  +-----+  
 *   | EC2 |  | EC2 |  | EC2 |  | EC2 |  
 *   +-----+  +-----+  +-----+  +-----+  
 *      |        |        |        |
 *      +--------+--------+--------+
 *                    |
 *                 +-----+
 *                 | ELB |
 *                 +-----+
 *                    |
 *       +------------+----------+
 *       |            |          |
 *   +--------+  +--------+  +--------+  
 *   | User-1 |  | User-1 |  | User-1 |  
 *   +--------+  +--------+  +--------+  
 * 
 * > Kya ye machines stateful ho sakti hai? If yes, then uska nuksaan kya hoga?
 * > Lets say humare paas 3 users hai: User-A, User-B, User-C. Agar User-C ne pehli
 *   request ki jo Instance-1  paas aai, and User-C ne Instance-1 pe kuch data store
 *   kr diya. Ab jab User-C dobara se call karega to kya uski request 100% guaranteed
 *   hai ki wo Instance-1 pe jaegi? No! Wo kahi par v jaa sakti hai.
 * > Agar request Instance-1 pe gyi to usko data mil jaega but agar uski request
 *   let's say Instance-3 pe gyi to usko data nhi milega, jisse inconsistency ho gyi.
 *   This was the problem with Stateful Connections.
 * 
 * Solution:
 * > To solve this problem, hum load balancers k andar kuch rules laga sakte hai.
 * > By default hum ek algorithm use krte hai called Round Robin Algorithm, which means
 *   pehli request pehle pe karo then 2nd request 2nd pe, then 3rd request third pe,
 *   then 4th request 4th then, then again 5th request 1st pe. Isse hum try krte hai
 *   ki jo routing hai wo kaafi linear ho.
 * > Humare paas ek chij aati hai called Sticky Sessions, means agar ek baar User-C ko
 *   koi particular Machine assign ho gyi to next time uski request guaranteed usi
 *   server k andar jaegi. To load balancer yha par ek state maintain karta hai ya
 *   fir wo cookies k andar v store kr skta hai.
 * 
 * Problem: But agar ye Instance-1 crash kr gyi to data can be lost.
 * 
 * Solution: 
 * > Agar humein true scaling karni hai to hm machines ko hum stateless rakhnge.
 * > Humein fark nhi padta ki kon sa user kis machine k saath connect kr rha hai,
 *   but agar hum data machine m nhi rkhnge to kaha rakhnge?
 * > Hum ek 'Central Server' bana lenge and we will make sure ki har DB Node isi k andar
 *   se data ko READ and WRITE karegi, jisse humein consistency mil gyi.
 * 
 *                (Database)
 *            +----------------+
 *            | Central Server |(Single Source of Truth)
 *            +----------------+
 *                    |
 *      +--------+--------+--------+
 *      |        |        |        |
 *   +-----+  +-----+  +-----+  +-----+  
 *   | EC2 |  | EC2 |  | EC2 |  | EC2 |  
 *   +-----+  +-----+  +-----+  +-----+  
 *      |        |        |        |
 *      +--------+--------+--------+
 *                    |
 *                 +-----+
 *                 | ELB |
 *                 +-----+
 *                    |
 *       +------------+----------+
 *       |            |          |
 *   +--------+  +--------+  +--------+  
 *   | User-1 |  | User-1 |  | User-1 |  
 *   +--------+  +--------+  +--------+  
 * 
 * > Ab user kisi v machine k andar jae, even if we add more machine, it doesn't matter,
 *   because we have a central server which is managing all the data. So, 4 DBs and
 *   future m agar aur machines aaenga to sabko same data milega, because now there is
 *   a single source of truth. And ye Data Machine data ko store krne k liye kuch v use 
 *   kr skti hai like Postgres, Oracle, etc.
 * > And at the end of the day, Database v ek server hai jo kisi EC2 Instance k andar
 *   chal rha hoga, bas Postgres is a way that how it keeps the data on the files.
 *   And iss instance ka v ek IP Address, PORT hoga and works in TCP, but isse hum
 *   public IP nhi denge because agar hum isse public IP denge to humara DB Internet pe
 *   expose ho jaega, isliye hum isse private network m rakhte hai so that sirf humari
 *   EC2 Machines isse access kr pae.
 * > Our problem is solved - our machines are stateless and koi v user aa sakta hai
 *   aur kisi v machine se connect kr skta hai and we don't have a problem because of
 *   the single source of truth.
*/

In [ ]:
/**
 * Real-World Problem:
 * > Ab humara application grow kr rha hai, to humne iss particular layers k upar 
 *   ek ASG laga diya. 
 * > AWS kehta hai ki agar hum machines ko automatically scale krna chahte hai to hum
 *   usse ek Automatically Scaling Group m daal de, and ASG k andar hum ek policy use
 *   krte hai jisme hum scale up and down k rules attach krte hai.
 * 
 *                    (Database)
 *                +----------------+
 *                | Central Server |(Single Source of Truth)
 *                +----------------+
 *                        |
 *          +--------+--------+--------+
 *          |        |        |        | 
 *    +-----|--------|--------|--------|-----+
 *    |  +-----+  +-----+  +-----+  +-----+  |
 *    |  | EC2 |  | EC2 |  | EC2 |  | EC2 |  | Auto-Scaling Groups
 *    |  +-----+  +-----+  +-----+  +-----+  |
 *    +-----|--------|--------|--------|-----+
 *          |        |        |        |     
 *          +--------+--------+--------+
 *                        |
 *                     +-----+
 *                     | ELB |
 *                     +-----+
 *                        |
 *           +------------+----------+
 *           |            |          |
 *       +--------+  +--------+  +--------+  
 *       | User-1 |  | User-1 |  | User-1 |  
 *       +--------+  +--------+  +--------+  
 * 
 * > Ab agar bahut saare users aa jae to kya mera ELB scale karega? 
 *   - Yes! Because ELB is managed by AWS.
 * > Ab agar bahut saare users aa jae to kya mere machines scale krengi?
 *   - Yes! Because maine uske upar ASG Configure kr rkha hai.
 * > But iss poore system m bottleneck kaha hai?
 *   - Database
 * 
 * > There is a service known as CloudWatch, wo inn saare EC2 Instances k resources ko
 *   monitor kr raha hai. Aur agar achanak se average response time spike krta hai
 *   10sec ka, jisse developers trigger ho gye, now what could be the problem?
 * > We'll ask:
 *   - Hum pehle Load Balancers check krnge
 *   - Fir EC2 Machines ki scaling check krnge
 *   - Then Database k logs check krnge
 * > Suppose humein milega ki Database ka CPU utilization bahut badha hua hai jiske karan
 *   se humara database slow ho gya hai and this is the bottleneck. And, because bahut
 *   saare READ and WRITES aa rhe hai, uske kaaran se humara DB slow ho gya hai.
 * 
 * Solution-1: Vertical Scaling
 * > Ab agar hum isse bahut saare resources de de to kya ye fast perform karega? Yes!
 * > Ya fir agar hm ek aur DB Server laga de to ye fast perform karega? Yes! But ek aur
 *   machine laana is not a good choice because meri consistency khatm ho jaegi.
 * > New machine is not a solution, so hm machine ko bada kr denge:
 *   - 2CPU -> 4CPU
 *   - 4GB RAM -> 16GB RAM
 * > Ab agar database ko monitor krnge to ye spike normal ho jaega.
 * > But ye jaise jaise users badhte jaenge to ye issue aate jaega aur humein fir se
 *   apne system ko vertically scale krna padega jiske wajah se:
 *   - Cost badhega
 *   - Jab hm machine ko upgrade krte hai, jabtak wo machine upgrade ho rhi hai tabtak
 *     humara database down jaata hai, jiske wajah se traffic m sudden dip hoga aur
 *     wo users wapas nhi aaenge, so at the end humein business loss hoga.
 * > Agar humein vertically scale krna hi hai to uske liye pre-planning lagta hai.
 *   For example, kal agar Amazon Sale hai to aaj raat 2 bje db down kr k vertically
 *   scale kr lenge, because 2bje raat m jyada traffic ni rehta. But still this is not
 *   a scalable solution, and agar mera ye single database instance down gya to mera
 *   poora application crash kr jaega jisse mera data loss hoga aur app v down jaega.
 * > So, we need some way jaha hum iss database ko v scale kar pae.
 *   
 * 
 * Solution-2: Horizontal Scaling Technique 1: Database Replication
 *   - It means we have one server i.e., Master/Primary DB (WRITE), aur hum iske bahut 
 *     saare READ replica/clone i.e., Slave/Secondary DB bana denge.
 *   - Fir humne ek rule banaya: Always write to Master jisse meri consistency wali
 *     problem dur ho jaegi, because agar saare servers Master DB k andar WRITE kr rhe
 *     hai, and DB k andar sirf one direction flow hai jisse DB load 50% kam ho gya.
 *   - And hum periodically iss data ko Replica-1 and Replica-2 m sync kr denge with
 *     slight delay. 
 * > Ab humara WRITE optimize ho gya jisse wo bottleneck nahi banega. And agar humara
 *   traffic bahut jyada badh rha hai to READ DB k bahut saare replicas bana denge
 *   jisse ek distributed system ban jaega, means humara database split ho gya hai.
 * > It's working perfectly fine, but iske v kuch issues hai:
 *   a. Increase in Cost
 *   b. Increase in Storage Consumption
 *   c. Data is not realtime
 * 
 * > Humein Youtube, Instagram jaise apps m views and likes realtime dikhane ki jarurt 
 *   nhi hai, agar data thoda delayed show hoga to v chalega.
 * > But Caption, Comments, Title, Description are realtime, so hum inko delay nahi 
 *   karna chahte because ye critical chije hai.
 * > So, application decide kr skta hai ki humein kya Primary se READ krna hai jo realtime
 *   m chahiye, but baaki chije jisme eventual consistency chalegi usse hum READ Replica
 *   se read krnge. And this is how we distribute the load, but agar WRITE down chala
 *   gya to single point of failure issue ab v hai, and iska solution is again: Quorum,
 *   jo aapas m voting kr k ek ko master promote kr leta hai and because humara data
 *   multiple jagha copied tha to data loss jyda nhi hota hai, but still there is a point
 *   WRITE DB overwhelm ho jae and down chala jae.
 * 
 * 
 * Solution-3: Horizontal Scaling Technique 2: Sharding + Hashing
 * > To solve the issue of Single Point of failure, we will split the database into
 *   multiple parts, so that single point of failure naa ho but isse data inconsistent 
 *   ho jaega.
 * > Inconsistency tabhi hota hai jab koi piece of data humne kisi aur DB m kiya aur
 *   read kisi aur DB m kr rhe hai. But agar hum ye data jiss db m write kre aur usi se 
 *   read krne ka try kre to data consistent ho jaega. So, we have to work on consistency.
 * > Hum har database ko ek index de dete hai aur wo saare indepdendent databases honge.
 * > Humne ek function likha aur iss db m jo data enter krna chahta hai uski userid 
 *   de denge:
 *   
 *   function mereLiyeIndexLeKarAao(userId) { 
 *      return Math.random(0 to 2)
 *   }
 * 
 * > Agar math.random ki value alag aayi to fir se inconsistent ho jaega because insert
 *   k time db-0 m insert hua, aur read krne k time db-1 or db-2 aa rha hai. To ye approach
 *   nahi chalega.
 * > We need a function jiska output humesa same rahe, to hum userId ko % kr denge mere
 *   numOfServers k saath jisse ye hoga ki hm jiss db m data insert krnge usi db se read
 *   krnge:
 * 
 *   function mereLiyeIndexLeKarAao(userId) { 
 *      return userId % 3 ====> 1 % 3 = idx-1
 *   }                          2 % 3 = idx-2
 *                              3 % 3 = idx-0 
 *                              4 % 3 = idx-1 
 *                              5 % 3 = idx-2
 * 
 * +---------------------------+ +---------------------------+ +---------------------------+
 * |                           | |                           | |                           |
 * |            +-----------+  | |            +-----------+  | |            +-----------+  |
 * |            | Read DB 1 |  | |            | Read DB 1 |  | |            | Read DB 1 |  |
 * | +----------+-----------+  | | +----------+-----------+  | | +----------+-----------+  |
 * | | Write DB | Read DB 2 |  | | | Write DB | Read DB 2 |  | | | Write DB | Read DB 2 |  |
 * | +----------+-----------+  | | +----------+-----------+  | | +----------+-----------+  |
 * |            | Read DB 3 |  | |            | Read DB 3 |  | |            | Read DB 3 |  |
 * |            +-----------+  | |            +-----------+  | |            +-----------+  |
 * |       Shard-0             | |       Shard-1             | |       Shard-2             |
 * +---------------------------+ +---------------------------+ +---------------------------+
 * 
 * > This becomes a round robin algorithm, which makes our data consistent.
 * > But Adding and Removing Databases still adds data inconsistency, which means
 *   suppose hum ek aur database shard add krte hai to numOfServers badh jaenge jo 3 se 4
 *   ho jaega. That means ab humara hash function badal gya: userId % 4
 *   - 1 % 3 = idx-1  ->  1 % 4 = 1
 *   - 2 % 3 = idx-2  ->  2 % 4 = 2
 *   - 3 % 3 = idx-0  ->  3 % 4 = 3 ----> Yaha se data inconsistent hona shuru hua
 *   - 4 % 3 = idx-1  ->  4 % 4 = 0
 *   - 5 % 3 = idx-2  ->  5 % 4 = 1
 *   - 6 % 3 = idx-0  ->  6 % 4 = 2
 *   - 7 % 3 = idx-1  ->  7 % 4 = 3
 * 
 * Solution-3: Horizontal Scaling Technique-3: Data Movement
 * > 3rd user ka data uthao aur copy kar do idx-0 to idx-3 mai.
 *   4th user ka data uthao aur copy kar do idx-1 to idx-0 mai.
 *   5th user ka data uthao aur copy kar do idx-2 to idx-1 mai.
 * > But data movement karna is not good approach, because isse overhead aa gya aur
 *   data loss k chances v hai.
 * > Humare paas 7 data hai, aur agar hum ek server add karte hai to humein 5 users ka
 *   data move krna pad rha hai, but what if ye data millions m ho, but data movement
 *   ki problem sirf tab aati hai jab OpenAI jaise companies rapidly scale krti hai which
 *   is very rare. So, we can rely on this system design but JOINS expensive ho gye hai 
 *   because humara adha data idhar hai aur adha udhar. Ans we have to go each cluster 
 *   and do application level JOINs.
 * 
 * Solution-4: Horizontal Scaling Technique-4: Consistent Hashing 
*/